In [98]:
import kagglehub

path = kagglehub.dataset_download("stanfordu/stanford-question-answering-dataset")

print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/stanfordu/stanford-question-answering-dataset/versions/2


In [99]:
import os

print(os.listdir(path))

['train-v1.1.json', 'dev-v1.1.json']


In [100]:
import os
import json
import pandas as pd
import kagglehub

# Download
path = kagglehub.dataset_download("stanfordu/stanford-question-answering-dataset")

# File path
file_path = os.path.join(path, "train-v1.1.json")

# Load data
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

questions = []
answers = []

for article in data["data"]:
    for para in article["paragraphs"]:
        for qa in para["qas"]:
            questions.append(qa["question"])
            answers.append(qa["answers"][0]["text"])

df = pd.DataFrame({
    "Question": questions,
    "Answer": answers
})

df.to_csv("squad_qa.csv", index=False)

print("CSV created successfully 🚀")

CSV created successfully 🚀


In [101]:
df.shape

(87599, 2)

In [103]:
df.head()

,Question,Answer
0,To whom did the Virgin Mary allegedly appear i...,Saint Bernadette Soubirous
1,What is in front of the Notre Dame Main Building?,a copper statue of Christ
2,The Basilica of the Sacred heart at Notre Dame...,the Main Building
3,What is the Grotto at Notre Dame?,a Marian place of prayer and reflection
4,What sits on top of the Main Building at Notre...,a golden statue of the Virgin Mary


In [104]:
# tokenize
def tokenize(text):
  text = text.lower()
  text = text.replace('?','')
  text = text.replace("'","")
  return text.split()

In [105]:
tokenize("What is in front of the Notre Dame Main Building?")

['what', 'is', 'in', 'front', 'of', 'the', 'notre', 'dame', 'main', 'building']

In [106]:
vocab = {'<UNK>': 0, '<PAD>': 1, '<SOS>': 2, '<EOS>': 3}

In [107]:
def build_vocab(row):
  tokenized_question = tokenize(row['Question'])
  tokenized_answer = tokenize(row['Answer'])

  merged_tokens = tokenized_question + tokenized_answer

  for token in merged_tokens:

    if token not in vocab:
      vocab[token] = len(vocab)

In [108]:
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
87594,None
87595,None
87596,None
87597,None


In [109]:
len(vocab)

72531

In [110]:
# convert words to numerical indices
def text_to_indices(text, vocab):

  indexed_text = []

  for token in tokenize(text):

    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [111]:
text_to_indices("What is in front of the Notre Dame Main Building?", vocab)

[19, 20, 12, 21, 22, 7, 23, 24, 25, 26]

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

In [113]:
class QADataset(Dataset):

  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):

    numerical_question = text_to_indices(self.df.iloc[index]['Question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['Answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [114]:
# use the full dataframe for training (no sampling/subset)
dataset = QADataset(df.reset_index(drop=True), vocab)
print('Total samples in df:', len(df))
print('Total samples in dataset:', len(dataset))
assert len(dataset) == len(df), 'Dataset is not using full df.'

Total samples in df: 87599
Total samples in dataset: 87599


In [115]:
def qa_collate_fn(batch):
  questions, answers = zip(*batch)

  # question lengths for packed RNNs
  q_lengths = torch.tensor([q.size(0) for q in questions], dtype=torch.long)
  padded_questions = pad_sequence(questions, batch_first=True, padding_value=1)  # PAD=1

  # answer sequences: add SOS (2) at start, EOS (3) at end
  answer_sequences = []
  for ans in answers:
    seq = [2] + ans.tolist() + [3]  # SOS + answer + EOS
    answer_sequences.append(torch.tensor(seq, dtype=torch.long))

  a_lengths = torch.tensor([len(seq) for seq in answer_sequences], dtype=torch.long)
  padded_answers = pad_sequence(answer_sequences, batch_first=True, padding_value=1)  # PAD=1

  return padded_questions.long(), q_lengths, padded_answers.long(), a_lengths

In [116]:
dataloader = DataLoader(
  dataset,
  batch_size=64,
  shuffle=True,
  collate_fn=qa_collate_fn
  )

In [117]:
for question, q_len, answer, a_len in dataloader:
  print('question shape:', question.shape)
  print('q_len shape:', q_len.shape)
  print('answer shape:', answer.shape)
  print('a_len shape:', a_len.shape)
  break

question shape: torch.Size([64, 24])
q_len shape: torch.Size([64])
answer shape: torch.Size([64, 20])
a_len shape: torch.Size([64])


In [132]:
import torch.nn as nn

In [139]:
class Attention(nn.Module):
  def __init__(self, hidden_dim):
    super().__init__()
    self.hidden_dim = hidden_dim
    self.v = nn.Linear(hidden_dim * 4, 1, bias=False)

  def forward(self, hidden, encoder_outputs, mask=None):
    batch_size = encoder_outputs.shape[0]
    max_len = encoder_outputs.shape[1]

    hidden_repeated = hidden.unsqueeze(1).repeat(1, max_len, 1)
    combined = torch.cat([hidden_repeated, encoder_outputs], dim=2)
    combined = torch.tanh(combined)
    attention_scores = self.v(combined).squeeze(-1)

    if mask is not None:
      attention_scores = attention_scores.masked_fill(mask == 0, float('-inf'))

    attention_weights = torch.softmax(attention_scores, dim=1).unsqueeze(1)
    context = torch.bmm(attention_weights, encoder_outputs).squeeze(1)
    return context, attention_weights

class Encoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_dim):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=1)
    self.gru = nn.GRU(embedding_dim, hidden_dim, bidirectional=True, batch_first=True)

  def forward(self, src, src_len):
    embedded = self.embedding(src)
    packed = pack_padded_sequence(embedded, src_len.cpu(), batch_first=True, enforce_sorted=False)
    outputs, hidden = self.gru(packed)
    outputs, _ = pad_packed_sequence(outputs, batch_first=True)
    hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
    return outputs, hidden

class Decoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_dim, attention):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=1)
    self.gru = nn.GRU(embedding_dim + hidden_dim * 2, hidden_dim * 2, batch_first=True)
    self.attention = attention
    self.fc_out = nn.Linear(hidden_dim * 4, vocab_size)

  def forward(self, target, hidden, encoder_outputs, src_mask=None):
    embedded = self.embedding(target).unsqueeze(1)
    context, attn_weights = self.attention(hidden, encoder_outputs, src_mask)
    gru_input = torch.cat([embedded.squeeze(1), context], dim=1).unsqueeze(1)
    output, hidden = self.gru(gru_input, hidden.unsqueeze(0))
    output_combined = torch.cat([output.squeeze(1), context], dim=1)
    prediction = self.fc_out(output_combined)
    return prediction, hidden.squeeze(0), attn_weights

class Seq2Seq(nn.Module):
  def __init__(self, vocab_size, embedding_dim=64, hidden_dim=96):
    super().__init__()
    self.attention = Attention(hidden_dim)
    self.encoder = Encoder(vocab_size, embedding_dim, hidden_dim)
    self.decoder = Decoder(vocab_size, embedding_dim, hidden_dim, self.attention)
    self.vocab_size = vocab_size

  def forward(self, src, src_len, tgt, teacher_forcing_ratio=0.5):
    encoder_outputs, hidden = self.encoder(src, src_len)
    src_mask = (src != 1).float()
    batch_size = tgt.shape[0]
    max_len = tgt.shape[1]
    outputs = []

    decoder_input = tgt[:, 0]
    for t in range(1, max_len):
      output, hidden, _ = self.decoder(decoder_input, hidden, encoder_outputs, src_mask)
      outputs.append(output.unsqueeze(1))
      teacher_force = torch.rand(1).item() < teacher_forcing_ratio
      decoder_input = tgt[:, t] if teacher_force else output.argmax(1)

    if len(outputs) == 0:
      return torch.zeros(batch_size, 0, self.vocab_size).to(src.device)
    
    outputs = torch.cat(outputs, dim=1)
    return outputs

In [140]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

# question tensor from first sample: [seq_len] -> [batch, seq_len]
a = dataset[0][0].long().unsqueeze(0)
print("shape of a:", a.shape)

b = x(a)
print("shape of b:", b.shape)

c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))
print("shape of e:", e.shape)

shape of a: torch.Size([1, 13])
shape of b: torch.Size([1, 13, 50])
shape of c: torch.Size([1, 13, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [141]:
learning_rate = 0.0015
epochs = 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Training device:', device)

Training device: cuda


In [142]:
model = Seq2Seq(len(vocab), embedding_dim=64, hidden_dim=96).to(device)

In [143]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [144]:
# training loop with Seq2Seq
model.train()

for epoch in range(epochs):
  total_loss = 0.0
  seen_samples = 0

  for question, q_len, answer, a_len in dataloader:
    question = question.to(device)
    q_len = q_len.to(device)
    answer = answer.to(device)
    a_len = a_len.to(device)

    optimizer.zero_grad()

    # forward pass with teacher forcing
    output = model(question, q_len, answer, teacher_forcing_ratio=0.5)

    # reshape for loss calculation
    output_flat = output.reshape(-1, model.vocab_size)
    target_flat = answer[:, 1:].reshape(-1)

    loss = criterion(output_flat, target_flat)

    loss.backward()
    optimizer.step()

    total_loss += loss.item()
    seen_samples += question.size(0)

  avg_loss = total_loss / len(dataloader)
  print(f"Epoch: {epoch+1}, Avg Loss: {avg_loss:.4f}, Seen: {seen_samples}/{len(dataset)}")

Epoch: 1, Avg Loss: 1.7975, Seen: 87599/87599
Epoch: 2, Avg Loss: 1.4237, Seen: 87599/87599
Epoch: 3, Avg Loss: 1.1183, Seen: 87599/87599
Epoch: 4, Avg Loss: 0.8703, Seen: 87599/87599
Epoch: 5, Avg Loss: 0.7833, Seen: 87599/87599
Epoch: 6, Avg Loss: 0.7083, Seen: 87599/87599
Epoch: 7, Avg Loss: 0.6282, Seen: 87599/87599
Epoch: 8, Avg Loss: 0.5720, Seen: 87599/87599
Epoch: 9, Avg Loss: 0.5257, Seen: 87599/87599
Epoch: 10, Avg Loss: 0.4703, Seen: 87599/87599


In [145]:
idx_to_token = {idx: token for token, idx in vocab.items()}

def predict(model, question, max_len=50):
  model.eval()
  numerical_question = text_to_indices(question, vocab)
  question_tensor = torch.tensor(numerical_question, dtype=torch.long).unsqueeze(0).to(device)
  q_len = torch.tensor([len(numerical_question)], dtype=torch.long).to(device)

  with torch.no_grad():
    encoder_outputs, hidden = model.encoder(question_tensor, q_len)
    src_mask = (question_tensor != 1).float()
    
    decoder_input = torch.tensor([2], dtype=torch.long).to(device)  # SOS token
    answer_tokens = []
    
    for _ in range(max_len):
      output, hidden, _ = model.decoder(decoder_input, hidden, encoder_outputs, src_mask)
      top_token = output.argmax(1).item()
      
      if top_token == 3:  # EOS token
        break
      
      if top_token > 3:  # Skip special tokens
        answer_tokens.append(idx_to_token.get(top_token, '<UNK>'))
      
      decoder_input = torch.tensor([top_token], dtype=torch.long).to(device)
  
  return ' '.join(answer_tokens) if answer_tokens else "I don't know"

In [ ]:
predict(model, "What is the name of the capital of USA?")

'green'

'green'

'green'

'green'